# 5SC28 Design Assignment — Section 4.2.2: PPO Reference Tracking

Extends the swing-up policy (4.2.1) to a **single policy** that:
1. Swings the disc up from the bottom (θ = 0°)
2. Tracks a reference θ_ref = π ± 15° around the top position

A switching controller is not valid — this is one end-to-end trained network.

**Approach**: augment the observation with [sin θ_ref, cos θ_ref] so the policy knows its target. The reference is sampled randomly at each episode reset. PPO is used (same as 4.2.1) for training stability.

---
# RUN ORDER — WHAT TO RUN FOR THE REAL HARDWARE

**REQUIRED, in this order:**
1. Cell 1 — install deps
2. Cell 2 — imports
3. Cell 4 — environment wrapper (defines `ALLOWED_VOLTAGES`, action + reward wrappers)
4. Cell 6 — loads/trains `model` (loads `ppo_reftrack.zip` if it exists, otherwise trains for a while)
5. **Cell 12 — HARDWARE DEMO.** Disc at bottom, USB connected, run this. Interrupt the kernel to stop.
6. Cell 13 — plots `hardware_log.npy` from the run above

**NOT needed for hardware (skip these):**
- Cells 8, 10 — simulation-only evaluation / plots
- Cells 14–16 — system-ID sim-vs-hardware diagnostic (separate experiment)
- Cell 17 — simulation-only test loop (infinite loop, never run on real hardware)
---

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'stable-baselines3', 'matplotlib'])


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /usr/bin/python3 -m pip install --upgrade pip


0

In [2]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

sys.path.insert(0, os.path.abspath('..'))
import gym_unbalanced_disk

/home/kaydenknapik/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
/home/kaydenknapik/.local/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/kaydenknapik/.local/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


## Reference-Tracking Environment Wrapper

The base environment has a 3D observation: [sin θ, cos θ, ω].

We extend it to 5D: **[sin θ, cos θ, ω, sin θ_ref, cos θ_ref]**

Using sin/cos for the reference (rather than the raw angle) avoids wrap-around discontinuities and keeps the representation consistent with the state encoding.

Reward: Gaussian centered on the reference angle:
$$r = \exp\!\left(-\frac{(\angle(\theta - \theta_\mathrm{ref}))^2}{2\sigma^2}\right)$$
where σ = π/8 ≈ 22.5°. This gives r ≈ 1 when on-target and decays smoothly away.

The reference is sampled uniformly from [π - 15°, π + 15°] at each episode reset.

In [3]:
ALLOWED_VOLTAGES = [-3, -2, -1, -0.5, -0.25, 0, 0.25, 0.5, 1, 2, 3]  # finer action set (Q-learning insight)
DT_SIM = 0.034  # matches measured real hardware control-loop time, not the nominal 0.025


class DiscretizeAction(gym.ActionWrapper):
    def __init__(self, env, voltages):
        super().__init__(env)
        self.voltages = np.array(voltages, dtype=np.float32)
        self.action_space = gym.spaces.Discrete(len(self.voltages))

    def action(self, act):
        return float(self.voltages[int(act)])


class UnbalancedDiskRef(gym.Wrapper):
    REF_RANGE  = np.radians(15)
    BRAKE_ZONE = np.pi / 4  # only penalise speed within 45 deg of target (Q-learning insight)

    def __init__(self, env, encoder_error_deg=2.0, dt=DT_SIM):
        super().__init__(env)
        low  = np.concatenate([env.observation_space.low,  [-1., -1.]]).astype(np.float32)
        high = np.concatenate([env.observation_space.high, [ 1.,  1.]]).astype(np.float32)
        self.observation_space = gym.spaces.Box(low=low, high=high, dtype=np.float32)
        self.th_ref = np.pi
        # Physically-derived sensor noise: sigma_omega from numerical-differentiation
        # error propagation of a noisy encoder reading sigma_theta (sqrt(2)/dt scaling).
        self.sigma_th = np.radians(encoder_error_deg)
        self.sigma_om = (np.sqrt(2) / dt) * self.sigma_th

    def _augment(self, raw_obs):
        ref = np.array([np.sin(self.th_ref), np.cos(self.th_ref)], dtype=np.float32)
        th  = np.arctan2(raw_obs[0], raw_obs[1]) + np.random.randn() * self.sigma_th
        om  = raw_obs[2] + np.random.randn() * self.sigma_om
        noisy = np.array([np.sin(th), np.cos(th), om], dtype=np.float32)
        return np.concatenate([noisy, ref])

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.th_ref = np.pi + np.random.uniform(-self.REF_RANGE, self.REF_RANGE)
        return self._augment(obs), info

    def step(self, action):
        obs, _, terminated, truncated, info = self.env.step(action)
        th     = np.arctan2(obs[0], obs[1])
        th_err = ((th - self.th_ref + np.pi) % (2 * np.pi)) - np.pi
        reward = 0.5 * (1.0 + np.cos(th_err))
        if abs(th_err) < self.BRAKE_ZONE:           # Q-learning-style conditional braking penalty
            reward -= 0.005 * obs[2]**2
        return self._augment(obs), float(reward), terminated, truncated, info


def make_env():
    env = gym.make('unbalanced-disk-sincos-v0', dt=DT_SIM)
    env = DiscretizeAction(env, ALLOWED_VOLTAGES)
    return UnbalancedDiskRef(env, dt=DT_SIM)


env = make_env()
print('obs space  :', env.observation_space)
print('action space:', env.action_space)
obs, _ = env.reset()
print('sample obs :', obs)


obs space  : Box([ -1.  -1. -40.  -1.  -1.], [ 1.  1. 40.  1.  1.], (5,), float32)
action space: Discrete(11)
sample obs : [-0.00810166  0.99996716  2.2775726   0.24690646 -0.9690393 ]


/home/kaydenknapik/.local/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:134: UserWarning: WARN: The obs returned by the `reset()` method was expecting numpy array dtype to be float32, actual type: float64
  logger.warn(
/home/kaydenknapik/.local/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


## PPO Training

Same PPO setup as the swing-up (4.2.1) — 4 parallel environments, 300k timesteps.
The only difference is the 5D observation and the reference-tracking reward.

If a saved model exists it is loaded directly to avoid retraining.

In [ ]:
_nb   = globals().get('__vsc_ipynb_file__', '')
_here = os.path.dirname(os.path.abspath(_nb)) if _nb else os.getcwd()
MODEL_PATH = os.path.join(_here, 'ppo_reftrack.zip')

if os.path.exists(MODEL_PATH):
    model = PPO.load(MODEL_PATH)
    print('Loaded saved model.')
else:
    vec_env = make_vec_env(make_env, n_envs=4)
    model = PPO(
        'MlpPolicy', vec_env,
        learning_rate=3e-4,
        gamma=0.99,
        ent_coef=0.005,
        n_steps=1024,
        batch_size=64,
        verbose=1,
        seed=42,
    )
    model.learn(total_timesteps=300_000)
    model.save(MODEL_PATH)
    print('Training done, model saved.')

Using cpu device


/home/kaydenknapik/.local/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:134: UserWarning: WARN: The obs returned by the `reset()` method was expecting numpy array dtype to be float32, actual type: float64
  logger.warn(
/home/kaydenknapik/.local/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
/home/kaydenknapik/.local/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:134: UserWarning: WARN: The obs returned by the `step()` method was expecting numpy array dtype to be float32, actual type: float64
  logger.warn(
/home/kaydenknapik/.local/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 300      |
|    ep_rew_mean     | 7.93     |
| time/              |          |
|    fps             | 2843     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 4096     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 300         |
|    ep_rew_mean          | 12.7        |
| time/                   |             |
|    fps                  | 2145        |
|    iterations           | 2           |
|    time_elapsed         | 3           |
|    total_timesteps      | 8192        |
| train/                  |             |
|    approx_kl            | 0.013376105 |
|    clip_fraction        | 0.205       |
|    clip_range           | 0.2         |
|    entropy_loss         | -2.39       |
|    explained_variance   | -0.241      |
|    learning_rate        | 0.

## Evaluation Across Reference Angles

Test the trained policy at several fixed reference angles: -15°, 0°, and +15° around the top.

In [ ]:
def run_episode(ref_deg, n_episodes=5):
    th_ref = np.pi + np.radians(ref_deg)
    rewards = []
    for _ in range(n_episodes):
        env_e = make_env()
        obs, _ = env_e.reset()
        env_e.th_ref = th_ref
        obs[-2] = np.sin(th_ref)
        obs[-1] = np.cos(th_ref)
        total = 0.0
        done = False
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, r, terminated, truncated, _ = env_e.step(action)
            total += r
            done = terminated or truncated
        rewards.append(total)
    return np.mean(rewards)

RUN_EVAL = False  # set to True to run simulation evaluation

if RUN_EVAL:
    for deg in [-15, -10, -5, 0, 5, 10, 15]:
        score = run_episode(deg)
        print(f'  ref = {deg:+3d}°  →  mean reward = {score:.1f} / 300')

## Episode Plots at Three Reference Angles

In [ ]:
def plot_episode(ref_deg):
    th_ref = np.pi + np.radians(ref_deg)
    env_e = make_env()
    obs, _ = env_e.reset()
    env_e.th_ref = th_ref
    obs[-2] = np.sin(th_ref)
    obs[-1] = np.cos(th_ref)

    obs_list, act_list, rew_list = [], [], []
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs_list.append(obs.copy())
        act_list.append(float(ALLOWED_VOLTAGES[int(action)]))
        obs, r, terminated, truncated, _ = env_e.step(action)
        rew_list.append(r)
        done = terminated or truncated

    obs_arr = np.array(obs_list)
    th_arr  = np.degrees(np.arctan2(obs_arr[:, 0], obs_arr[:, 1]))
    t       = np.arange(len(th_arr)) * DT_SIM

    fig, axes = plt.subplots(3, 1, figsize=(9, 6), sharex=True)
    axes[0].plot(t, th_arr, label='θ')
    axes[0].axhline(180 + ref_deg, color='r', ls='--', label=f'ref ({180+ref_deg}°)')
    axes[0].set_ylabel('θ [deg]'); axes[0].legend(); axes[0].grid(True)
    axes[1].plot(t, obs_arr[:, 2], 'g')
    axes[1].set_ylabel('ω [rad/s]'); axes[1].grid(True)
    axes[2].plot(t, act_list, 'r')
    axes[2].set_ylabel('u [V]'); axes[2].set_xlabel('time [s]'); axes[2].grid(True)
    plt.suptitle(f'PPO reference tracking — ref = {180+ref_deg}°')
    plt.tight_layout()
    plt.show()
    print(f'Total reward: {sum(rew_list):.1f} / {len(rew_list)}')

if RUN_EVAL:
    for deg in [-15, 0, 15]:
        plot_episode(deg)

## >>> RUN THIS CELL FOR THE REAL HARDWARE <<<

**Required first:** cells 1, 2, 4, 6 must already have been run (installs, imports, wrapper, `model`).

Make sure the disc is at the **bottom position** before running. Connects over USB, swings up to 180°, holds for 1.5s, then tracks a ±15° sine wave reference. Interrupt the kernel to stop.

In [ ]:
import time
import pygame

env_live = gym_unbalanced_disk.UnbalancedDisk_exp_sincos()
obs, _ = env_live.reset()

def make_obs(raw_obs, th_ref):
    ref = np.array([np.sin(th_ref), np.cos(th_ref)], dtype=np.float32)
    return np.concatenate([np.asarray(raw_obs, dtype=np.float32), ref])

def draw_overlay(env, phase, th_ref, th):
    if not hasattr(env, '_font'):
        pygame.font.init()
        env._font = pygame.font.SysFont('monospace', 18, bold=True)
    lines = [
        f'Phase:  {phase}',
        f'Target: {np.degrees(th_ref) % 360:.1f} deg',
        f'Angle:  {np.degrees(th) % 360 if abs(np.degrees(th) % 360 - 360) > 0.5 else 0.0:.1f} deg',
    ]
    for i, line in enumerate(lines):
        surf = env._font.render(line, True, (30, 30, 30))
        env.viewer.blit(surf, (10, 10 + i * 22))
    pygame.display.flip()

AMP           = np.radians(15)
PERIOD        = 6.0
HOLD_TOP_SECS = 1.5
RENDER_EVERY  = 0  # 0 = no render on hardware (keeps control loop at ~25ms)

phase       = 'swing-up'
phase_start = time.time()
log    = []
t_last = time.time()
step   = 0

print('Phase: swing-up (target = 180°)')
try:
    while True:
        t_now     = time.time()
        actual_dt = t_now - t_last
        t_last    = t_now
        th        = np.arctan2(obs[0], obs[1])
        near_top  = abs((th - np.pi + np.pi) % (2*np.pi) - np.pi) < np.radians(25)

        if phase == 'swing-up' and near_top:
            phase, phase_start = 'holding', t_now
            print('Phase: holding at 180°...')
        elif phase == 'holding' and (t_now - phase_start) >= HOLD_TOP_SECS:
            phase, phase_start = 'tracking', t_now
            print('Phase: tracking sine wave ±15°')

        th_ref = np.pi + AMP * np.sin(2 * np.pi * (t_now - phase_start) / PERIOD) if phase == 'tracking' else np.pi

        action, _ = model.predict(make_obs(obs, th_ref), deterministic=True)
        u_applied = ALLOWED_VOLTAGES[int(action)]
        log.append([t_now, np.degrees(th) % 360 if abs(np.degrees(th) % 360 - 360) > 0.5 else 0.0, float(obs[2]), u_applied, actual_dt, np.degrees(th_ref) % 360])

        obs, _, terminated, truncated, _ = env_live.step(u_applied)
        step += 1

        if RENDER_EVERY > 0 and step % RENDER_EVERY == 0:
            env_live.render()
            draw_overlay(env_live, phase, th_ref, th)

        if terminated or truncated:
            print('Episode ended, restarting...')
            obs, _ = env_live.reset()
            phase, phase_start = 'swing-up', time.time()
            t_last = time.time()
            step   = 0
            print('Phase: swing-up (target = 180°)')
finally:
    env_live.close()
    if log:
        log_arr = np.array(log)
        np.save('hardware_log.npy', log_arr)
        print(f'\nLog saved ({len(log)} steps)')
        print(f'Mean dt: {log_arr[:,4].mean()*1000:.1f}ms  (target: 25ms)')
        print(f'Omega std: {log_arr[:,2].std():.3f} rad/s')
        print(f'Max |omega|: {np.abs(log_arr[:,2]).max():.1f} rad/s')

In [ ]:
## Plot hardware log: angle vs target over time
if os.path.exists('hardware_log.npy'):
    hw = np.load('hardware_log.npy')
    # columns: t, theta_deg, omega, action, dt, th_ref_deg
    t       = hw[:, 0] - hw[0, 0]
    th_deg  = hw[:, 1]
    th_ref_deg = hw[:, 5] if hw.shape[1] > 5 else np.full(len(t), 180.0)

    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

    axes[0].plot(t, th_deg,     label='θ (actual)',  color='steelblue')
    axes[0].plot(t, th_ref_deg, label='θ_ref (target)', color='crimson', linestyle='--', linewidth=1.5)
    axes[0].set_ylabel('Angle (deg)')
    axes[0].legend(); axes[0].grid(True)
    axes[0].set_title('Hardware run: angle tracking')

    axes[1].plot(t, hw[:, 2], color='darkorange', label='ω (rad/s)')
    axes[1].axhline(0, color='k', linewidth=0.5)
    axes[1].set_ylabel('ω (rad/s)'); axes[1].legend(); axes[1].grid(True)

    axes[2].plot(t, hw[:, 3], color='green', label='action (V)')
    axes[2].set_ylabel('Action (V)'); axes[2].set_xlabel('Time (s)')
    axes[2].legend(); axes[2].grid(True)

    plt.tight_layout()
    plt.savefig('hardware_log_plot.png', dpi=120)
    plt.show()
    print(f'Steps: {len(t)}  Duration: {t[-1]:.1f}s  Mean dt: {hw[:,4].mean()*1000:.1f}ms')
else:
    print('No hardware_log.npy found — run the hardware cell first.')


In [ ]:
## System ID: record sim trajectory (policy actions + resulting states)
env_id  = make_env()
obs, _  = env_id.reset()
sim_rec = []  # [step_t, theta_deg, omega, action]

for step in range(300):
    action, _ = model.predict(obs, deterministic=True)
    u_applied = ALLOWED_VOLTAGES[int(action)]
    th = np.arctan2(obs[0], obs[1])
    sim_rec.append([step * env_id.unwrapped.dt, np.degrees(th) % 360, float(obs[2]), u_applied])
    obs, _, term, trunc, _ = env_id.step(action)
    if term or trunc:
        break

env_id.close()
sim_rec = np.array(sim_rec)
np.save('sysid_sim.npy', sim_rec)
print(f'Sim: {len(sim_rec)} steps, max|omega|={np.abs(sim_rec[:,2]).max():.1f}, final theta={sim_rec[-1,1]:.1f} deg')


In [ ]:
## System ID: play sim actions open-loop on hardware, record response
import time
sim_rec = np.load('sysid_sim.npy')

env_hw = gym_unbalanced_disk.UnbalancedDisk_exp_sincos()
obs, _ = env_hw.reset()
hw_rec = []  # [actual_t, theta_deg, omega, action]
t0 = time.time()

for i, row in enumerate(sim_rec):
    action = np.array([row[3]], dtype=np.float32)
    th = np.arctan2(obs[0], obs[1])
    hw_rec.append([time.time() - t0, np.degrees(th) % 360, float(obs[2]), float(row[3])])
    obs, _, _, _, _ = env_hw.step(action)

env_hw.close()
hw_rec = np.array(hw_rec)
np.save('sysid_hw.npy', hw_rec)
print(f'Hardware: {len(hw_rec)} steps, actual duration={hw_rec[-1,0]:.2f}s (sim expected {sim_rec[-1,0]:.2f}s)')
print(f'Mean dt: {np.diff(hw_rec[:,0]).mean()*1000:.1f}ms  (sim: {sim_rec[1,0]*1000:.1f}ms)')


In [ ]:
## System ID: compare sim vs hardware response to identical actions
sim_rec = np.load('sysid_sim.npy')
hw_rec  = np.load('sysid_hw.npy')

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=False)

# Plot vs step number to remove timing difference — shows pure dynamics mismatch
steps = np.arange(min(len(sim_rec), len(hw_rec)))
n = len(steps)

axes[0].plot(steps, sim_rec[:n, 1], label='Sim θ', color='steelblue')
axes[0].plot(steps, hw_rec[:n,  1], label='HW θ',  color='crimson', linestyle='--')
axes[0].set_ylabel('Angle (deg)'); axes[0].legend(); axes[0].grid(True)
axes[0].set_title('Same actions, step-aligned — difference = dynamics mismatch')

axes[1].plot(steps, sim_rec[:n, 2], label='Sim ω', color='steelblue')
axes[1].plot(steps, hw_rec[:n,  2], label='HW ω',  color='crimson', linestyle='--')
axes[1].set_ylabel('ω (rad/s)'); axes[1].legend(); axes[1].grid(True)

axes[2].plot(steps, sim_rec[:n, 3], label='Action (V)', color='green')
axes[2].set_ylabel('Action (V)'); axes[2].set_xlabel('Step')
axes[2].legend(); axes[2].grid(True)

plt.tight_layout()
plt.savefig('sysid_comparison.png', dpi=120)
plt.show()

# Timing comparison
print(f'Sim step:      {sim_rec[1,0]*1000:.1f}ms')
print(f'HW mean step:  {np.diff(hw_rec[:,0]).mean()*1000:.1f}ms')
print(f'Timing ratio:  {np.diff(hw_rec[:,0]).mean() / sim_rec[1,0]:.3f}x (ideal=1.0)')


In [ ]:
## Simulation test with obs vs target logging
import time

def make_obs(raw_obs, th_ref):
    ref = np.array([np.sin(th_ref), np.cos(th_ref)], dtype=np.float32)
    return np.concatenate([np.asarray(raw_obs, dtype=np.float32), ref])

env_sim = gym_unbalanced_disk.UnbalancedDisk_sincos(dt=DT_SIM)
obs, _  = env_sim.reset()
phase, phase_start = 'swing-up', time.time()
sim_log = []

try:
    while True:
        t_now    = time.time()
        th       = np.arctan2(obs[0], obs[1])
        near_top = abs((th - np.pi + np.pi) % (2*np.pi) - np.pi) < np.radians(25)
        if phase == 'swing-up' and near_top:
            phase, phase_start = 'holding', t_now
        elif phase == 'holding' and (t_now - phase_start) >= 1.5:
            phase, phase_start = 'tracking', t_now
        th_ref = np.pi + np.radians(15) * np.sin(2*np.pi*(t_now - phase_start) / 6.0) if phase == 'tracking' else np.pi
        action, _ = model.predict(make_obs(obs, th_ref), deterministic=True)
        u_applied = ALLOWED_VOLTAGES[int(action)]
        sim_log.append([t_now, np.degrees(th) % 360 if abs(np.degrees(th) % 360 - 360) > 0.5 else 0.0, float(obs[2]), u_applied, np.degrees(th_ref) % 360])
        obs, _, terminated, truncated, _ = env_sim.step(u_applied)
        env_sim.render()
        time.sleep(env_sim.unwrapped.dt)
        if terminated or truncated:
            obs, _ = env_sim.reset()
            phase, phase_start = 'swing-up', time.time()
finally:
    env_sim.close()
    if sim_log:
        sl = np.array(sim_log)
        t  = sl[:, 0] - sl[0, 0]
        fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
        axes[0].plot(t, sl[:, 1], label='θ (actual)',      color='steelblue')
        axes[0].plot(t, sl[:, 4], label='θ_ref (target)',  color='crimson', linestyle='--', linewidth=1.5)
        axes[0].set_ylabel('Angle (deg)'); axes[0].legend(); axes[0].grid(True)
        axes[0].set_title('Simulation: angle tracking')
        axes[1].plot(t, sl[:, 2], color='darkorange', label='ω (rad/s)')
        axes[1].axhline(0, color='k', linewidth=0.5)
        axes[1].set_ylabel('ω (rad/s)'); axes[1].legend(); axes[1].grid(True)
        axes[2].plot(t, sl[:, 3], color='green', label='action (V)')
        axes[2].set_ylabel('Action (V)'); axes[2].set_xlabel('Time (s)')
        axes[2].legend(); axes[2].grid(True)
        plt.tight_layout()
        plt.savefig('sim_log_plot.png', dpi=120)
        plt.show()